In [1]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd()
candidates = [root, root.parent, root.parent.parent]
for base in candidates:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

from efgp_eigenpro_py.eigenpro_precond import build_preconditioner

np.set_printoptions(precision=3, suppress=True)

have_plot = False
plt = None


def enable_plotting():
    global have_plot, plt
    try:
        import importlib
        plt = importlib.import_module("matplotlib.pyplot")
        have_plot = True
        print("matplotlib enabled")
    except BaseException as exc:
        have_plot = False
        plt = None
        print("matplotlib not usable, skip plots:", exc)


In [2]:
import numpy as np, scipy, matplotlib
from scipy.sparse.linalg import cg
print("numpy", np.__version__)
print("scipy", scipy.__version__)
print("matplotlib", matplotlib.__version__)
print("cg import ok")

numpy 1.26.4
scipy 1.15.3
matplotlib 3.10.8
cg import ok


In [3]:
def make_problem(n=200, q=10, case="gap", seed=0, complex_case=False):
    rng = np.random.default_rng(seed)
    if complex_case:
        Z = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
        Q, _ = np.linalg.qr(Z)
    else:
        Z = rng.normal(size=(n, n))
        Q, _ = np.linalg.qr(Z)

    if case == "mild":
        lam = 10.0 ** (-4.0 * np.arange(n) / (n - 1))
    elif case == "gap":
        lam = np.empty(n, dtype=float)
        lam[:q] = 10.0 ** np.linspace(6.0, 4.0, q)
        lam[q:] = 10.0 ** np.linspace(2.0, 0.0, n - q)
    elif case == "flat":
        lam = np.linspace(100.0, 1.0, n)
    else:
        raise ValueError("unknown case")

    A = Q @ np.diag(lam) @ Q.conj().T
    return A, Q, lam


def spectral_checks(A, Q, lam, q, precond):
    Uq = Q[:, :q]
    mu = lam[q]

    err_top = []
    for i in range(q):
        ui = Uq[:, i]
        got = precond(A @ ui)
        ref = mu * ui
        err_top.append(np.linalg.norm(got - ref) / max(np.linalg.norm(ref), 1e-15))

    err_tail = []
    for i in range(q, Q.shape[1]):
        ui = Q[:, i]
        got = precond(A @ ui)
        ref = lam[i] * ui
        err_tail.append(np.linalg.norm(got - ref) / max(np.linalg.norm(ref), 1e-15))

    kappa_A = lam[0] / lam[-1]
    kappa_PA = lam[q] / lam[-1]

    print("max flatten err (top) =", float(np.max(err_top)))
    print("max flatten err (tail) =", float(np.max(err_tail)))
    print("cond(A) =", float(kappa_A))
    print("cond(PA) ideal =", float(kappa_PA))


In [4]:
def richardson_history(matvec, b, x0, eta, maxiter, precond=None):
    x = x0.copy()
    r = matvec(x) - b
    hist = [np.linalg.norm(r) / max(np.linalg.norm(b), 1e-15)]

    for _ in range(maxiter):
        z = precond(r) if precond is not None else r
        Az = matvec(z)
        x = x - eta * z
        r = r - eta * Az
        hist.append(np.linalg.norm(r) / max(np.linalg.norm(b), 1e-15))
    return x, np.array(hist)


def pcg_history(matvec, b, tol, maxiter, precond=None):
    try:
        from scipy.sparse.linalg import LinearOperator, cg  # noqa: F401
    except Exception as exc:
        print("scipy not usable, skip pcg:", exc)
        return None, None, None

    n = b.size
    dtype = np.result_type(b.dtype, np.complex128)
    Aop = LinearOperator((n, n), matvec=matvec, dtype=dtype)
    Mop = None if precond is None else LinearOperator((n, n), matvec=precond, dtype=dtype)

    hist = []

    def cb(xk):
        rk = matvec(xk) - b
        hist.append(np.linalg.norm(rk) / max(np.linalg.norm(b), 1e-15))

    x, info = cg(Aop, b, rtol=tol, atol=0.0, maxiter=maxiter, M=Mop, callback=cb)
    if len(hist) == 0:
        rk = matvec(x) - b
        hist.append(np.linalg.norm(rk) / max(np.linalg.norm(b), 1e-15))
    return x, np.array(hist), info


In [5]:
def run_experiment(case="gap", n=200, q=10, seed=0, complex_case=False, maxiter=200):
    A, Q, lam = make_problem(n=n, q=q, case=case, seed=seed, complex_case=complex_case)

    def matvec(v):
        return A @ v

    rng = np.random.default_rng(seed + 123)
    if complex_case:
        b = rng.normal(size=n) + 1j * rng.normal(size=n)
    else:
        b = rng.normal(size=n)
    x_ref = np.linalg.solve(A, b)

    Uq = Q[:, :q]
    theta = lam[:q]
    mu = lam[q]
    P = build_preconditioner(theta, Uq, mu).apply

    spectral_checks(A, Q, lam, q, P)

    eta_plain = 0.95 * 2.0 / (lam[0] + lam[-1])
    eta_prec = 0.95 * 2.0 / (mu + lam[-1])

    x0 = np.zeros_like(b)
    xr, hr = richardson_history(matvec, b, x0, eta_plain, maxiter=maxiter, precond=None)
    xrp, hrp = richardson_history(matvec, b, x0, eta_prec, maxiter=maxiter, precond=P)

    xcg, hcg, info_cg = pcg_history(matvec, b, tol=1e-12, maxiter=maxiter, precond=None)
    xcgp, hcgp, info_pcg = pcg_history(matvec, b, tol=1e-12, maxiter=maxiter, precond=P)

    def relerr(x):
        return np.linalg.norm(x - x_ref) / max(np.linalg.norm(x_ref), 1e-15)

    print("plain Richardson relerr =", float(relerr(xr)))
    print("prec  Richardson relerr =", float(relerr(xrp)))

    if xcg is not None:
        print("plain CG relerr =", float(relerr(xcg)), "info =", info_cg)
    if xcgp is not None:
        print("prec  CG relerr =", float(relerr(xcgp)), "info =", info_pcg)

    if have_plot:
        plt.figure(figsize=(6, 4))
        plt.semilogy(hr, label="Richardson")
        plt.semilogy(hrp, label="Precond Richardson")
        if hcg is not None:
            plt.semilogy(np.arange(1, len(hcg) + 1), hcg, label="CG")
        if hcgp is not None:
            plt.semilogy(np.arange(1, len(hcgp) + 1), hcgp, label="Precond CG")
        plt.xlabel("iteration")
        plt.ylabel("relative residual")
        plt.title(f"case={case}, n={n}, q={q}")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("matplotlib not available, skip plots")

    return {
        "hr": hr,
        "hrp": hrp,
        "hcg": hcg,
        "hcgp": hcgp,
    }


In [6]:
print("=== case: gap (real) ===")
run_experiment(case="gap", n=200, q=10, seed=0, complex_case=False, maxiter=200)

print("=== case: mild (real) ===")
run_experiment(case="mild", n=200, q=10, seed=1, complex_case=False, maxiter=200)

print("=== case: flat (real) ===")
run_experiment(case="flat", n=200, q=10, seed=2, complex_case=False, maxiter=200)

print("=== case: gap (complex) ===")
run_experiment(case="gap", n=200, q=10, seed=3, complex_case=True, maxiter=200)


=== case: gap (real) ===
max flatten err (top) = 2.3414437580510167e-12
max flatten err (tail) = 4.3994034375165463e-11
cond(A) = 1000000.0
cond(PA) ideal = 100.0
plain Richardson relerr = 0.9992636554385853
prec  Richardson relerr = 0.009289006079665802
plain CG relerr = 2.738619315756729e-05 info = 200
prec  CG relerr = 9.870321639169793e-12 info = 0
matplotlib not available, skip plots
=== case: mild (real) ===
max flatten err (top) = 4.672484924932661e-16
max flatten err (tail) = 1.7435717369413833e-12
cond(A) = 10000.0
cond(PA) ideal = 6294.9889902218865
plain Richardson relerr = 0.9349649076281498
prec  Richardson relerr = 0.9005281512815856
plain CG relerr = 0.0037695842196773916 info = 200
prec  CG relerr = 0.0007229802567884429 info = 200
matplotlib not available, skip plots
=== case: flat (real) ===
max flatten err (top) = 7.04811099436996e-16
max flatten err (tail) = 4.772189133580658e-14
cond(A) = 100.0
cond(PA) ideal = 95.0251256281407
plain Richardson relerr = 0.013244431

{'hr': array([1.   , 0.992, 0.989, 0.988, 0.987, 0.986, 0.986, 0.985, 0.985,
        0.985, 0.984, 0.984, 0.984, 0.983, 0.983, 0.983, 0.983, 0.982,
        0.982, 0.982, 0.982, 0.982, 0.982, 0.981, 0.981, 0.981, 0.981,
        0.981, 0.981, 0.981, 0.98 , 0.98 , 0.98 , 0.98 , 0.98 , 0.98 ,
        0.98 , 0.98 , 0.98 , 0.98 , 0.979, 0.979, 0.979, 0.979, 0.979,
        0.979, 0.979, 0.979, 0.979, 0.979, 0.979, 0.979, 0.979, 0.978,
        0.978, 0.978, 0.978, 0.978, 0.978, 0.978, 0.978, 0.978, 0.978,
        0.978, 0.978, 0.978, 0.978, 0.978, 0.978, 0.978, 0.977, 0.977,
        0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977,
        0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977, 0.977,
        0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976,
        0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976, 0.976,
        0.976, 0.976, 0.976, 0.976, 0.975, 0.975, 0.975, 0.975, 0.975,
        0.975, 0.975, 0.975, 0.975, 0.975, 0.975, 0.975, 0.975, 0.975,
